In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from huggingface_hub import login
from dotenv import load_dotenv
from openai import OpenAI
import logging
import os
from tqdm import tqdm
import json
import time
import fitz
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
load_dotenv()
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()

# Configure the huggingface_hub logger to suppress warnings
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# Login using the token
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

logging.getLogger("huggingface_hub").setLevel(logging.INFO)

## Create a Specific Prompt for Japanese Text Extraction
Let's create a specialized prompt that asks the model to:

1. Ignore furigana
2. Preserve structure
3. Output in JSON format

In [ ]:
def get_prompt():
    return """
    This image contains Japanese text from a textbook with furigana (small kana above kanji).
    
    Task:
    1. Analyze the image and extract the main Japanese text
    2. IGNORE all furigana (small kana characters above kanji)
    3. Preserve correct sentence structure and punctuation
    4. Output the text in JSON format with this structure:
       {
         "chapter": "chapter_number",
         "exercises": [
           {
             "number": "exercise_number",
             "sentences": ["sentence1", "sentence2", ...]
           }
         ]
       }
    
    If chapter or exercise numbers aren't visible, use null values.
    Return ONLY the JSON output, nothing else.
    """

## Using LLaVa-Next Model
https://huggingface.co/docs/transformers/model_doc/llava_next

In [ ]:
# Load model and processor
model_id = "llava-hf/llama3-llava-next-8b-hf"  # You can also try the larger variant
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaNextForConditionalGeneration.from_pretrained(
    model_id, 
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model-00001-of-00004.safetensors:   6%|6         | 315M/4.96G [00:00<?, ?B/s]

Download complete. Moving file to C:\Users\Linus\.cache\huggingface\hub\models--llava-hf--llama3-llava-next-8b-hf\blobs\8b9f1f1855068947b0de7434fd486ae4c00c2df21cf0c758989d0866632795aa


model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

In [ ]:
def load_quantized_llava(quant_4bit=True, quant_8bit=True):
    """Load LLaVA-NeXT model with 8-bit quantization"""
    model_id = "llava-hf/llama3-llava-next-8b-hf"
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )
    
    # Install required packages if they don't exist
    try:
        import bitsandbytes
    except ImportError:
        print("Installing bitsandbytes...")
        !pip install -q bitsandbytes
        import bitsandbytes
    
    # Load processor normally
    processor = AutoProcessor.from_pretrained(model_id)
    
    # Load model with 8-bit quantization
    if quant_4bit:
        model = LlavaNextForConditionalGeneration.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            low_cpu_mem_usage=True
        )
    else if quant_8bit:
        model = LlavaNextForConditionalGeneration.from_pretrained(
            model_id,
            load_in_8bit=True,  # Enable 8-bit quantization
            device_map="auto",
            torch_dtype=torch.float16
        )
    else:
        model = LlavaNextForConditionalGeneration.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16
        )
        
    return model, processor

In [ ]:
def process_image_with_llava(image_path, prompt, model, processor):
    # Load image
    if image_path.startswith('http'):
        image = Image.open(requests.get(image_path, stream=True).raw)
    else:
        image = Image.open(image_path)
    
    # Process inputs
    inputs = processor(prompt, image, return_tensors="pt").to(model.device)
    
    # Generate response
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False
        )
    
    # Decode and return response
    response = processor.decode(output[0], skip_special_tokens=True)
    return response.strip()

## Using GPT-4o for Comparison

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def process_with_gpt(image_path, prompt, model="GPT4-mini"):
    try:
        response = openai.ChatCompletion.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                    ]
                }
            ]
        )
        
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error with {model_name}: {str(e)}")
        return None

## Process PDF Pages

In [ ]:
def convert_pdf_to_images(pdf_path, output_dir):
    """Convert PDF pages to images"""
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    page_images = []
    
    for page_num in tqdm(range(len(doc)), desc="Converting PDF pages to images"):
        page = doc[page_num]
        
        # Convert page to image (2x zoom for better quality)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_path = f"{output_dir}/page_{page_num}.png"
        pix.save(img_path)
        page_images.append((page_num, img_path))
    
    return page_images

In [ ]:
def process_page(args):
    """Process a single page with selected models"""
    page_num, img_path, models = args
    results = {}
    
    # Japanese text extraction prompt (same for all models)
    prompt = get_prompt()
    
    # Process with LLaVA if enabled
    if "llava" in models and models["llava"]:
        model, processor = models["llava"]
        llava_result = process_image_with_llava(img_path, prompt, model, processor)
        try:
            llava_json = json.loads(llava_result)
            llava_json["page_number"] = page_num + 1
            results["llava"] = llava_json
        except (json.JSONDecodeError, TypeError) as e:
            print(f"Could not parse LLaVA JSON from page {page_num+1}: {str(e)}")
            results["llava"] = {"error": str(e), "raw": llava_result[:200] + "..." if llava_result else None}
    
    # Process with GPT-4o if enabled
    if "gpt4o" in models and models["gpt4o"]:
        # Add a delay to avoid rate limiting
        time.sleep(1)
        gpt4o_result = process_with_openai(img_path, prompt, "gpt-4o")
        try:
            gpt4o_json = json.loads(gpt4o_result)
            gpt4o_json["page_number"] = page_num + 1
            results["gpt4o"] = gpt4o_json
        except (json.JSONDecodeError, TypeError) as e:
            print(f"Could not parse GPT-4o JSON from page {page_num+1}: {str(e)}")
            results["gpt4o"] = {"error": str(e), "raw": gpt4o_result[:200] + "..." if gpt4o_result else None}
    
    # Process with GPT-4o mini if enabled
    if "gpt4o_mini" in models and models["gpt4o_mini"]:
        # Add a delay to avoid rate limiting
        time.sleep(1)
        mini_result = process_with_openai(img_path, "gpt-4o-mini")
        try:
            mini_json = json.loads(mini_result)
            mini_json["page_number"] = page_num + 1
            results["gpt4o_mini"] = mini_json
        except (json.JSONDecodeError, TypeError) as e:
            print(f"Could not parse GPT-4o mini JSON from page {page_num+1}: {str(e)}")
            results["gpt4o_mini"] = {"error": str(e), "raw": mini_result[:200] + "..." if mini_result else None}
    
    return results

In [ ]:
def process_pdf_multimodel(pdf_path, output_dir, use_llava=True, use_gpt4o=False, use_gpt4o_mini=False, max_workers=4):
    """Process a PDF with multiple models in parallel"""
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # First, convert PDF to images
    page_images = convert_pdf_to_images(pdf_path, output_dir)
    
    # Load models as needed
    models = {}
    openai = OpenAI()
    
    if use_llava:
        print("Loading LLaVA-NeXT model...")
        models["llava"] = load_quantized_llava()
        print("LLaVA-NeXT model loaded successfully")
    
    if use_gpt4o:
        models["gpt4o"] = True
    
    if use_gpt4o_mini:
        models["gpt4o_mini"] = True
    
    # Process pages in parallel
    process_args = [(page_num, img_path, models) for page_num, img_path in page_images]
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(process_page, process_args),
            total=len(process_args),
            desc="Processing pages with models"
        ))
    
    # Organize results by model
    organized_results = {
        "llava": [],
        "gpt4o": [],
        "gpt4o_mini": []
    }
    
    for page_result in results:
        for model_name in organized_results.keys():
            if model_name in page_result and "error" not in page_result[model_name]:
                organized_results[model_name].append(page_result[model_name])
    
    # Save raw results for each model
    for model_name, model_results in organized_results.items():
        if model_results:
            with open(f"{output_dir}/{model_name}_results.json", "w", encoding="utf-8") as f:
                json.dump(model_results, f, ensure_ascii=False, indent=2)
    
    return organized_results

## Validation and Post-Processing
Add a validation step to ensure the JSON output is correctly formatted:

In [ ]:
def validate_and_clean_results(results, model_name):
    """Clean and merge results by chapter"""
    cleaned_results = []
    
    for page_result in results:
        if isinstance(page_result, dict) and "exercises" in page_result:
            cleaned_results.append(page_result)
        else:
            print(f"Skipping invalid {model_name} result for page {page_result.get('page_number', 'unknown')}")
    
    # Merge results by chapter
    merged_results = {}
    for result in cleaned_results:
        chapter = result.get("chapter")
        if chapter not in merged_results:
            merged_results[chapter] = {"chapter": chapter, "exercises": []}
        
        # Append exercises
        merged_results[chapter]["exercises"].extend(result.get("exercises", []))
    
    return list(merged_results.values())

In [ ]:
def compare_model_results(model_results_dict):
    """Compare results between different models"""
    comparison = {
        "summary": {},
        "details": []
    }
    
    # Get list of models with results
    models = [model for model, results in model_results_dict.items() if results]
    
    if len(models) <= 1:
        return {"message": "Need at least two models with results to compare"}
    
    # Add summary stats for each model
    for model in models:
        results = model_results_dict[model]
        comparison["summary"][f"{model}_chapters"] = len(results)
        comparison["summary"][f"{model}_exercises"] = sum(len(chapter.get("exercises", [])) for chapter in results)
    
    # Find chapters that exist in all models
    chapter_sets = {model: {chapter.get("chapter") for chapter in model_results_dict[model]} for model in models}
    common_chapters = set.intersection(*chapter_sets.values())
    comparison["summary"]["common_chapters"] = len(common_chapters)
    
    # Compare common chapters in detail
    chapter_data = {
        model: {chapter.get("chapter"): chapter for chapter in model_results_dict[model]}
        for model in models
    }
    
    for chapter_id in common_chapters:
        chapter_comparison = {"chapter": chapter_id}
        
        # Get exercises from each model
        exercises_by_model = {
            model: {ex.get("number"): ex for ex in chapter_data[model][chapter_id].get("exercises", [])}
            for model in models
        }
        
        # Find common exercises
        exercise_sets = {model: set(exs.keys()) for model, exs in exercises_by_model.items()}
        common_exercises = set.intersection(*exercise_sets.values())
        
        chapter_comparison["common_exercises"] = len(common_exercises)
        for model in models:
            chapter_comparison[f"{model}_exercises"] = len(exercises_by_model[model])
        
        # Compare sentences for common exercises
        differences = []
        for ex_num in common_exercises:
            sentences_by_model = {
                model: exercises_by_model[model][ex_num].get("sentences", [])
                for model in models
            }
            
            # Check if all models have the same sentences
            first_model = models[0]
            reference_sentences = sentences_by_model[first_model]
            
            has_differences = False
            for model in models[1:]:
                if sentences_by_model[model] != reference_sentences:
                    has_differences = True
                    break
            
            if has_differences:
                diff_entry = {"exercise": ex_num}
                for model in models:
                    diff_entry[f"{model}_sentences"] = sentences_by_model[model]
                differences.append(diff_entry)
        
        chapter_comparison["differences"] = differences
        comparison["details"].append(chapter_comparison)
    
    return comparison

# Full Pipeline
Putting it all together

In [ ]:
def extract_japanese_text_from_pdf(
   pdf_path, 
   output_dir="output", 
   use_llava=True, 
   use_gpt4o=False, 
   use_gpt4o_mini=False, 
   openai_api_key=None,
   max_workers=4
):
   """Main function to extract Japanese text from PDF with multiple models"""
   print(f"Processing PDF: {pdf_path}")
   
   # Process PDF with selected models
   raw_results = process_pdf_multimodel(
       pdf_path, 
       output_dir, 
       use_llava=use_llava, 
       use_gpt4o=use_gpt4o, 
       use_gpt4o_mini=use_gpt4o_mini,
       openai_api_key=openai_api_key,
       max_workers=max_workers
   )
   
   # Validate and clean results for each model
   final_results = {}
   
   for model_name, model_results in raw_results.items():
       if model_results:
           final_results[model_name] = validate_and_clean_results(model_results, model_name)
           
           # Save final structured results
           with open(f"{output_dir}/final_{model_name}_structured.json", "w", encoding="utf-8") as f:
               json.dump(final_results[model_name], f, ensure_ascii=False, indent=2)
   
   # Compare results if multiple models were used
   if len(final_results) > 1:
       comparison = compare_model_results(final_results)
       with open(f"{output_dir}/model_comparison.json", "w", encoding="utf-8") as f:
           json.dump(comparison, f, ensure_ascii=False, indent=2)
       
       print(f"Model comparison saved to {output_dir}/model_comparison.json")
   
   print(f"Processing complete. Results saved to {output_dir}/")
   return final_results

# Using it

## Using all the models

In [ ]:
# Example Jupyter notebook usage

# Install required packages
!pip install -q transformers accelerate bitsandbytes torch pillow pymupdf tqdm

# Process a PDF with multiple models
results = extract_japanese_text_from_pdf(
    pdf_path="your_textbook.pdf",
    output_dir="japanese_extraction_output",
    use_llava=True,              # Use LLaVA-NeXT
    use_gpt4o=True,              # Use GPT-4o
    use_gpt4o_mini=True,         # Use GPT-4o mini
    openai_api_key=os.environ["OPENAI_API_KEY"],
    max_workers=4                # Adjust based on your system
)

# Examine the results
for model, data in results.items():
    print(f"\n{model} extracted {len(data)} chapters")
    for chapter in data:
        print(f"  Chapter {chapter['chapter']}: {sum(len(ex.get('sentences', [])) for ex in chapter['exercises'])} sentences")

#### If you want to try with just one model at a time (e.g., just LLaVA):

In [ ]:
# Use only LLaVA-NeXT
llava_results = extract_japanese_text_from_pdf(
    pdf_path="your_textbook.pdf",
    output_dir="llava_only_output",
    use_llava=True,
    use_gpt4o=False,
    use_gpt4o_mini=False
)

#### Test with a subset of pages first

In [ ]:
# Test with just a few pages
pdf_path = "your_textbook.pdf"
doc = fitz.open(pdf_path)
test_pdf_path = "test_sample.pdf"

# Create a sample PDF with first 3 pages
doc.select([0, 1, 2])  # Select first 3 pages
doc.save(test_pdf_path)
doc.close()

# Process the sample
sample_results = extract_japanese_text_from_pdf(
    pdf_path=test_pdf_path,
    output_dir="sample_test_output",
    use_llava=True,
    use_gpt4o_mini=True,  # Less expensive than full GPT-4o
)

## Visualizing Results

In [ ]:
# Load results
def load_and_analyze_results(output_dir):
    models = ["llava", "gpt4o", "gpt4o_mini"]
    results = {}
    
    # Load structured results for each model
    for model in models:
        try:
            with open(f"{output_dir}/final_{model}_structured.json", "r", encoding="utf-8") as f:
                results[model] = json.load(f)
        except FileNotFoundError:
            print(f"No results for {model}")
    
    # Create analysis dataframe
    analysis = []
    for model, data in results.items():
        total_exercises = 0
        total_sentences = 0
        
        for chapter in data:
            chapter_exercises = len(chapter.get("exercises", []))
            chapter_sentences = sum(len(ex.get("sentences", [])) for ex in chapter.get("exercises", []))
            
            total_exercises += chapter_exercises
            total_sentences += chapter_sentences
            
            analysis.append({
                "model": model,
                "chapter": chapter.get("chapter"),
                "exercises": chapter_exercises,
                "sentences": chapter_sentences
            })
    
    return pd.DataFrame(analysis)

# Generate visualization
def visualize_model_comparison(output_dir):
    df = load_and_analyze_results(output_dir)
    
    if df.empty:
        print("No data available for visualization")
        return
    
    # Create summary by model
    summary = df.groupby("model").sum().reset_index()
    
    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    
    # Exercise count
    summary.plot(kind="bar", x="model", y="exercises", ax=ax1)
    ax1.set_title("Total Exercises Extracted")
    ax1.set_ylabel("Exercise Count")
    
    # Sentence count  
    summary.plot(kind="bar", x="model", y="sentences", ax=ax2)
    ax2.set_title("Total Sentences Extracted")
    ax2.set_ylabel("Sentence Count")
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/model_comparison_chart.png")
    plt.show()
    
    return summary

# Use in notebook
results_summary = visualize_model_comparison("japanese_extraction_output")
print(results_summary)

### Checking Quality of Furigana Handling

In [ ]:
def check_furigana_handling(output_dir, sample_size=5):
    """Sample sentences and check for furigana characters"""
    # Common furigana patterns
    furigana_patterns = {
        "hiragana": "ぁあぃいぅうぇえぉおかがきぎくぐけげこごさざしじすずせぜそぞただちぢっつづてでとどなにぬねのはばぱひびぴふぶぷへべぺほぼぽまみむめもゃやゅゆょよらりるれろゎわゐゑをんゔゕゖ",
        "small_kana": "ぁぃぅぇぉっゃゅょゎ"
    }
    
    # Load results from each model
    models = ["llava", "gpt4o", "gpt4o_mini"]
    all_sentences = {}
    
    for model in models:
        try:
            with open(f"{output_dir}/final_{model}_structured.json", "r", encoding="utf-8") as f:
                data = json.load(f)
                sentences = []
                
                for chapter in data:
                    for exercise in chapter.get("exercises", []):
                        sentences.extend(exercise.get("sentences", []))
                
                # Randomly sample sentences
                import random
                if len(sentences) > sample_size:
                    sentences = random.sample(sentences, sample_size)
                
                all_sentences[model] = sentences
        except FileNotFoundError:
            print(f"No results for {model}")
    
    # Check for furigana patterns
    results = []
    for model, sentences in all_sentences.items():
        for sentence in sentences:
            # Count small kana characters (often used for furigana)
            small_kana_count = sum(sentence.count(char) for char in furigana_patterns["small_kana"])
            
            results.append({
                "model": model,
                "sentence": sentence,
                "length": len(sentence),
                "small_kana_count": small_kana_count,
                "small_kana_ratio": small_kana_count / len(sentence) if len(sentence) > 0 else 0
            })
    
    return pd.DataFrame(results)

# Use in notebook
furigana_analysis = check_furigana_handling("japanese_extraction_output")
print(furigana_analysis.groupby("model").mean()[["small_kana_count", "small_kana_ratio"]])